<a href="https://colab.research.google.com/github/williamlidberg/Analyses-of-Environmental-Data-2/blob/main/modules/module_4/assignment_4_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # Analyze satellite imagery with Geemap Updated for 2026
Geemap is a Python tool that makes it easy for beginners to create interactive maps using Google Earth Engine (GEE). Google Earth Engine is like a powerful online library of satellite images and geographic data stored in the cloud. People in the field of geography and environmental science widely use it.

Google Earth Engine has tools for programming in both JavaScript and Python. While the JavaScript tools have lots of helpful documentation and an interactive editor, the Python tools have less documentation and are not as user-friendly for visualizing results.

That's where Geemap comes in. Geemap is a Python package that was made to bridge this gap. It uses ipyleaflet and ipywidgets to help you analyze and visualize data from Google Earth Engine in a more beginner-friendly way. You can easily work with geographic datasets and create interactive maps right in a Jupyter notebook. It's a handy tool for anyone wanting to explore and understand Earth observation data without diving too deep into complex coding.

 # Installation
Geemap is preinstalled with Google Colab, but you can install it using PyPI or Anaconda as well:

`pip install geemap`  
`conda install geemap -c conda-forge`

Read the documentation for more info regarding installation on different systems: https://courses.geemap.org/geemap_intro/02_installation/#key-features

 ### Task 1 Setup
To use geemap, you must first sign up for a Google Earth Engine account. You cannot use Google Earth Engine unless your application has been approved. Once you receive the application approval email, the next step is to run the code below and grant access for Google Colab to use your Google account and Google Earth Engine. If everything works as planned, you should see an interactive map.

In [ ]:
# Authenticate and initialize Google Earth Engine API
import ee
ee.Authenticate()
ee.Initialize(project='ee-williamlidberg') # Please change ee-williamlidberg to your own user

 To check if everything is working, run these lines. They should output a map.

In [ ]:
import geemap

Map = geemap.Map()
Map.add_basemap("Esri.WorldImagery")
Map

If you want a different basemap you can select one from the list.

In [ ]:
for basemap in geemap.basemaps.keys():
    print(basemap)

 If you want to center the map on somewhere specific, you can set the coordinates and zoom level with `.setCenter()`. Note that the coordinates are in longitude/latitude. When you right-click in Google Maps, you get the coordinates in latitude/longitude. If you want to translate the coordinates from Google Maps, simply switch the numbers around, so `64.2, 20` becomes `20, 64.2`.

In [ ]:
Map = geemap.Map()
Map.layers = Map.layers[1:]
Map.add_tile_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    name="Esri Dark Gray",
    attribution="Esri",
)
Map.setCenter(20, 64.2, 5)
Map

 # Working with image collections
Google Earth Engine comes with a lot of large image series that you can use for your projects. The [Sentinel-2 constellation](https://sentinels.copernicus.eu/web/sentinel/missions/sentinel-2/news/-/asset_publisher/Ac0d/content/sentinel-2-images-the-globe-every-5-days;jsessionid=2BC4823847466509FA382BB7ED954C47.jvm1?redirect=https%3A%2F%2Fsentinels.copernicus.eu%2Fweb%2Fsentinel%2Fmissions%2Fsentinel-2%2Fnews%3Bjsessionid%3D2BC4823847466509FA382BB7ED954C47.jvm1%3Fp_p_id%3D101_INSTANCE_Ac0d%26p_p_lifecycle%3D0%26p_p_state%3Dnormal%26p_p_mode%3Dview%26p_p_col_id%3Dcolumn-1%26p_p_col_count%3D1%26_101_INSTANCE_Ac0d_cur%3D4%26_101_INSTANCE_Ac0d_keywords%3D%26_101_INSTANCE_Ac0d_advancedSearch%3Dfalse%26_101_INSTANCE_Ac0d_delta%3D30%26_101_INSTANCE_Ac0d_andOperator%3Dtrue) allows every spot on Earth to be revisited every five days with the same viewing direction. However, the northern latitudes are not imaged during the dark winter months.

Sentinel-2 consists of 2 satellites. First came Sentinel-2A, which was launched in 2015. Next came Sentinel-2B in 2017.

Two additional satellites (Sentinel-2C and 2D) were launched in 2024, making a total of four Sentinel-2 satellites.

Sentinel-2 has many bands, but the ones we are interested in in this module will be B2, B3, B4, and B8: Blue, green, red, and Near Infrared.

| Band | Resolution | Wavelength | Description                        |
|------|------------|------------|------------------------------------|
| B1   | 60 m       | 443 nm     | Ultra Blue (Coastal and Aerosol)   |
| B2   | 10 m       | 490 nm     | Blue                               |
| B3   | 10 m       | 560 nm     | Green                              |
| B4   | 10 m       | 665 nm     | Red                                |
| B5   | 20 m       | 705 nm     | Visible and Near Infrared (VNIR)  |
| B6   | 20 m       | 740 nm     | Visible and Near Infrared (VNIR)  |
| B7   | 20 m       | 783 nm     | Visible and Near Infrared (VNIR)  |
| B8   | 10 m       | 842 nm     | Visible and Near Infrared (VNIR)  |
| B8a  | 20 m       | 865 nm     | Visible and Near Infrared (VNIR)  |
| B9   | 60 m       | 940 nm     | Short Wave Infrared (SWIR)        |
| B10  | 60 m       | 1375 nm    | Short Wave Infrared (SWIR)        |
| B11  | 20 m       | 1610 nm    | Short Wave Infrared (SWIR)        |
| B12  | 20 m       | 2190 nm    | Short Wave Infrared (SWIR)        |


 We can also limit the collection in space and time.

 ### Select all images from last summer in Umeå

In [ ]:
# Set the coordinates for umeå as a boundig box.
Umeå = ee.Geometry.Polygon(
    [[[20.0695, 63.8608],
      [20.3986, 63.8608],
      [20.3986, 63.5922],
      [20.0695, 63.5922]]])

# Set the time interval
start_date = '2025-06-01'
end_date = '2025-07-31'

sentinel2 = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
             .filterBounds(Umeå)
             .filterDate(start_date, end_date)
             .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
)

sentinel2

 Since multiple images will overlap in this time and region, we need to choose which images we want to display. One way to do it is to select the images with the least cloud coverage.

In [ ]:
# Get the least cloudy image
image = sentinel2.sort('CLOUDY_PIXEL_PERCENTAGE').first()

# Get the date of the image
image_date = image.get('system:time_start')
print("Image Date:", ee.Date(image_date).format('YYYY-MM-dd').getInfo())


 Let's inspect Umeå during a clear summer day. This plots the least cloudy image in the collection.

In [ ]:
# Display the image on the map

Map.add_basemap("Esri.WorldStreetMap")
Map.setCenter(20.2630, 63.8258, 11) # This is the coordinates in long/lat and the zoom level

Map.addLayer(image, {
    'bands': ['B4', 'B3', 'B2'], # red, green, blue
    'min': 0,
    'max': 3000,
    'gamma': 1.4
}, 'RGB')

# Display the map
Map.addLayerControl()
Map

 ### Task 2
a) Why do you only see an image on the left side of Umeå?

b) Use what you have learned so far and answer the question. Could you ski in Umeå in December 2023?

 # NDVI
NDVI stands for "Normalized Difference Vegetation Index." It is a vegetation index that quantifies the presence and health of vegetation in a given area based on the difference in reflectance between the near-infrared and red bands of satellite or aerial imagery. NDVI values typically range from -1 to 1, where higher values indicate healthier and denser vegetation.

Let's have a closer look at Gotland. First in color. Gotland is too large to fit into one image, so we can create a mosaic dataset. Note that this is a special kind of mosaic where each pixel is selected separately from all available images within the time window 2026-06-01 to 2026-09-30.

In [ ]:
import ee
import geemap

Gotland = ee.Geometry.Polygon([
    [18.743, 57.908],  # NW corner
    [19.726, 57.908],  # NE corner
    [19.726, 57],       # SE corner
    [18.743, 57]        # SW corner
])

START_DATE = '2026-06-01'
END_DATE = '2026-09-30'

# Set cloud filtering parameters
CLOUD_FILTER = 10  # Maximum cloud cover percentage
CLD_PRB_THRESH = 10  # Cloud probability threshold
NIR_DRK_THRESH = 0.15  # Near-infrared dark threshold

sentinel2 = (ee.ImageCollection('COPERNICUS/S2')
             .filterBounds(Gotland)
             .filterDate(ee.Date(START_DATE), ee.Date(END_DATE))
             .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
             .map(lambda image: image.updateMask(image.select(['B8']).gt(NIR_DRK_THRESH)))
            )

# Create a cloud-free composite
composite = sentinel2.reduce(ee.Reducer.median())

Map = geemap.Map()
Map.add_basemap("Esri.WorldStreetMap")
Map.addLayer(composite, {
    'bands': ['B4_median', 'B3_median', 'B2_median'], # This selects the median value from all cloud free pixels.
    'min': 0,
    'max': 3000,
    'gamma': 1.4
}, 'RGB Composite')
Map.setCenter(18.4, 57.5, 10)
Map.addLayerControl()
Map


 NDVI can be calculated by adding this line of code to the previous code: `ndvi = composite.normalizedDifference(['B8_median', 'B4_median'])`

In [ ]:
import ee
import geemap

Gotland = ee.Geometry.Polygon([
    [18.743, 57.908],  # NW corner
    [19.726, 57.908],  # NE corner
    [19.726, 57],       # SE corner
    [18.743, 57]        # SW corner
])

START_DATE = '2026-06-01'
END_DATE = '2026-09-30'
CLOUD_FILTER = 10  # Maximum cloud cover percentage
CLD_PRB_THRESH = 10  # Cloud probability threshold
NIR_DRK_THRESH = 0.15  # Near-infrared dark threshold

sentinel2 = (ee.ImageCollection('COPERNICUS/S2')
             .filterBounds(Gotland)
             .filterDate(ee.Date(START_DATE), ee.Date(END_DATE))
             .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
             .map(lambda image: image.updateMask(image.select(['B8']).gt(NIR_DRK_THRESH)))
            )

composite = sentinel2.reduce(ee.Reducer.median())

# Calculate NDVI
ndvi = composite.normalizedDifference(['B8_median', 'B4_median'])

Map = geemap.Map()
Map.add_basemap("Esri.WorldStreetMap")
Map.addLayer(ndvi, {
    'palette': ['blue', 'white', 'green'],  # Adjust the palette from RGB to something more fitting NDVI
    'min': -1,
    'max': 1
}, 'NDVI')

Map.setCenter(18.4, 57.5, 15)
Map.addLayerControl()
Map


 ### Task 3
Calculate the NDVI for your home town/village (or some other location you know well) for the summer of 2018.

 # Combine satellite images with other data
To analyze the data in satellite imagery, we often need to extract it to features we are interested in. It can be field plots, forest stands, wetlands, etc. In this case, we will use wetlands from Gotland which have been classified based on their "nature value". Start by downloading the wetland polygons as shapefiles from Naturvårdsverket.

In [ ]:
!wget https://geodata.naturvardsverket.se/nedladdning/VMI/ursprunglig_digitalisering/I_Gotland_VMI.zip
!unzip /content/I_Gotland_VMI.zip

 Inspect the data in the shapefile.

In [ ]:
import geopandas as gpd
gdf = gpd.read_file("/content/I_Gotland_VMI_Ytor.shp")
gdf

 This dataset has some issues with naming where Swedish letters like å, ö, and ä are used. Instead of inspecting all columns and fixing issues, we can just select the column we are interested in and drop the rest.

In [ ]:
gdf = gdf[gdf.geom_type == 'Polygon'] # drop all fatures that are not polygons
gdf = gdf[['NVKLASS', 'geometry']].copy() # copy the columns to keep
# 1. Fix invalid polygons
gdf['geometry'] = gdf['geometry'].buffer(0)
gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.0001, preserve_topology=True)

# --- Clean geometries for Earth Engine ---
def clean_geom(g):
    if g is None or g.is_empty:
        return None
    if not g.is_valid:
        g = g.buffer(0)
    if g.geom_type == "Polygon":
        ext = [(float(round(x,6)), float(round(y,6))) for x,y,*_ in g.exterior.coords if abs(x)<=180 and abs(y)<=90]
        return Polygon(ext) if len(ext)>=4 else None
    if g.geom_type == "MultiPolygon":
        polys = [p for p in g.geoms if p.is_valid and not p.is_empty]
        return MultiPolygon(polys) if polys else None
    return None


# Translate the swedish names to english names to avoid symbols like å, ö ,ä.
value_mapping = {'Låga naturvärden':'Low nature values','Vissa naturvärden':'Some nature values', 'Högt naturvärde':'High Nature value', 'Mycket högt naturvärde': 'Very high nature values'}
gdf['NVKLASS'] = gdf['NVKLASS'].replace(value_mapping)
#gdf = gdf.iloc[:100, :]
#save to new shapefile
gdf.to_file("/content/I_Gotland_VMI_Ytor_clean.shp")

 Now we can load the cleaned shapefile into geemap by converting it to a format that Google Earth Engine can handle using `shp_to_ee`. In order to do that, you first need to install `pycrs`.

In [ ]:
!pip install pycrs

 Let's start by just overlaying the polygons on a map to see if they look ok after our data cleaning.

In [ ]:
import geemap
import geopandas as gpd

Map = geemap.Map()
Map.layers = Map.layers[1:]
Map.add_tile_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    name="Esri Dark Gray",
    attribution="Esri",
)

wetlands_shp = "/content/I_Gotland_VMI_Ytor_clean.shp"

gdf = gpd.read_file(wetlands_shp)
gdf['geometry'] = gdf['geometry'].buffer(0)
gdf = gdf.to_crs('EPSG:4326')

Map.add_gdf(
    gdf,
    layer_name="Wetlands",
    style={
        'color': '#00FFFF',
        'fillColor': '#00FFFF',
        'fillOpacity': 0.3,
        'weight': 2
    }
)

Map.setCenter(18.4, 57.5, 9)
Map.addLayerControl()
Map

 # Extract raster values to polygons
Now you have seen how to extract NDVI and how to import polygons into geemap. Let's combine the two and extract NDVI values for these wetlands for further analysis. Start by creating a mosaic of NDVI for Gotland. This may take some time.

In [ ]:
import geopandas as gpd, geemap
from shapely.geometry import Polygon, MultiPolygon

gdf = gpd.read_file("/content/I_Gotland_VMI_Ytor_clean.shp").to_crs("EPSG:4326")


# # --- Clean geometries for Earth Engine ---
def clean_geom(g):
    if g is None or g.is_empty:
        return None
    if not g.is_valid:
        g = g.buffer(0)
    if g.geom_type == "Polygon":
        ext = [(float(round(x,6)), float(round(y,6))) for x,y,*_ in g.exterior.coords if abs(x)<=180 and abs(y)<=90]
        return Polygon(ext) if len(ext)>=4 else None
    if g.geom_type == "MultiPolygon":
        polys = [p for p in g.geoms if p.is_valid and not p.is_empty]
        return MultiPolygon(polys) if polys else None
    return None

gdf["geometry"] = gdf["geometry"].apply(clean_geom)

# --- Convert to Earth Engine ---
wetlands_ee = geemap.geopandas_to_ee(gdf)

# calculate NDVI
sentinel2 = (ee.ImageCollection('COPERNICUS/S2')
             .filterBounds(Gotland)
             .filterDate(ee.Date(START_DATE), ee.Date(END_DATE))
             .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
             .map(lambda image: image.updateMask(image.select(['B8']).gt(NIR_DRK_THRESH)))
            )
composite = sentinel2.reduce(ee.Reducer.median())
ndvi = composite.normalizedDifference(['B8_median', 'B4_median'])
ndvi_stats = ndvi.reduceRegions(
    collection=wetlands_ee,
    reducer=ee.Reducer.mean(),
    scale=10
)
features = ndvi_stats.getInfo()['features']
ndvi_gdf = gpd.GeoDataFrame.from_features(features)

 We can now properly analyze the data. Here is a quick plot of mean NDVI on wetlands split by nature value.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))
sns.violinplot(x='NVKLASS', y='mean', data=ndvi_gdf, palette='Dark2')
plt.title('NDVI from wetlands on Gotland from summer 2026')
plt.xlabel('Nature Class')
plt.ylabel('Mean NDVI')
plt.show()


 # Time series
One cool thing with satellites is the temporal aspect, which allows us to follow environmental change over time. This requires a bit more work, but let's walk through it. Let's limit the analysis to 500 randomly sampled wetlands to save time.

In [ ]:
import geopandas as gpd
gdf = gpd.read_file("/content/I_Gotland_VMI_Ytor_clean.shp")
sampled_geodf = gdf.sample(n=500, random_state=42)
sampled_geodf.to_file("/content/sampled_wetlands.shp")

 Next, let's start with the first half of the code, which is the same as before.

In [ ]:
import ee
import geemap
import geopandas as gpd

# Define Gotland geometry
Gotland = ee.Geometry.Polygon([
  [18.743, 57.908],  # NW corner
  [19.726, 57.908],  # NE corner
  [19.726, 57],      # SE corner
  [18.743, 57]       # SW corner
])

START_YEAR = 2023
CLOUD_FILTER = 20

def filter_monthly_sentinel2(month):
    """Filters Sentinel-2 data for a specific month"""
    start_date = ee.Date(f'{START_YEAR}-{month:02d}-01')
    end_date = start_date.advance(1, 'month')

    def mask_clouds(image):
        opaque = image.select('MSK_CLASSI_OPAQUE').eq(0)
        cirrus = image.select('MSK_CLASSI_CIRRUS').eq(0)
        return image.updateMask(opaque.And(cirrus))

    return (
        ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
        .filterBounds(Gotland)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
        .map(mask_clouds)
    )

# Load wetland shapefile
wetlands_shp = "/content/sampled_wetlands.shp"
gdf = gpd.read_file(wetlands_shp).to_crs("EPSG:4326")
gdf["geometry"] = gdf["geometry"].apply(clean_geom)   # needs clean_geom defined earlier
wetlands_ee = geemap.geopandas_to_ee(gdf)

 Next, you need to specify which months you are interested in and then loop over the NDVI mosaics for each month. This step takes a few minutes.

In [ ]:
start_month = 1
end_month = 12
ndvi_features = []

for month in range(start_month, end_month + 1):
    monthly_images = filter_monthly_sentinel2(month)

    n = monthly_images.size().getInfo()
    if n == 0:
        print(f"{START_YEAR}-{month:02d}: no images, skipping")
        continue

    composite = monthly_images.select(['B8', 'B4']).median()
    ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')

    month_label = f"{START_YEAR}-{month:02d}"
    stats = ndvi.reduceRegions(
        collection=wetlands_ee,
        reducer=ee.Reducer.mean(),
        scale=10,
    ).map(lambda f: f.set('month', month_label))

    ndvi_features.extend(stats.getInfo()['features'])
    print(f"{month_label}: {n} images")

ndvi_gdf = gpd.GeoDataFrame.from_features(ndvi_features, crs="EPSG:4326")
ndvi_gdf

 Now we have a dataframe to analyze further. Time series of NDVI values can be used as input for machine learning models to extract spatiotemporal patterns from the data.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))
sns.boxplot(x="month", y="mean", hue='NVKLASS', data=ndvi_gdf, palette='Dark2')
plt.title('NDVI in wetlands with very high nature value')
plt.xlabel('Month')
plt.ylabel('Mean NDVI')
plt.show()

 Since this is temporal data, we can reuse some code from assignment three to create animated plots. Start by installing `chart_studio` from assignment 3.

In [ ]:
!pip install chart_studio

 Notice how the shapes change over time, and pay attention to the behavior of wetlands with low nature values compared to the others.

In [ ]:
import pandas as pd
import plotly.express as px

# Drop the geometry column (not needed for plotting) and remove empty results
plot_df = pd.DataFrame(ndvi_gdf.drop(columns="geometry")).dropna(subset=["mean"])

# Fixed orders keep the x-axis, colors and animation slider stable across frames
months = sorted(plot_df["month"].unique())
classes = sorted(plot_df["NVKLASS"].astype(str).unique())
plot_df["NVKLASS"] = plot_df["NVKLASS"].astype(str)

fig = px.violin(
    plot_df,
    x="NVKLASS",
    y="mean",
    color="NVKLASS",
    box=True,                      # small box plot inside each violin
    animation_frame="month",
    category_orders={"month": months, "NVKLASS": classes},
    range_y=[-0.2, 1],
    labels={"mean": "Mean NDVI", "NVKLASS": "Wetland class", "month": "Month"},
    template="plotly_dark",
)
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 2000
fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 0
fig.update_layout(showlegend=False, title="Monthly NDVI per wetland class")
fig.show()

 ### Task 4
Select another part of Sweden and see if the wetlands behave differently from wetlands on Gotland based on NDVI values.

You will find other parts of Sweden here: https://geodata.naturvardsverket.se/nedladdning/VMI/ursprunglig_digitalisering/

If the data size is too big, you can randomly sample fewer wetlands. Remember that we used about 500 wetlands so far. QGIS is a good companion to notebooks. It's always a good habit to inspect the data you are working with.

 # Change detection

In [ ]:
import ee
import geemap

Map = geemap.Map(center=[57.4, 18.9], zoom=8)
Map.layers = Map.layers[1:]
Map.add_tile_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    name="Esri Dark Gray",
    attribution="Esri",
)

Gotland = ee.Geometry.Rectangle([18.0, 56.8, 19.5, 58.0])

START_1, END_1 = '2020-06-01', '2020-08-31'
START_2, END_2 = '2025-06-01', '2025-08-31'
CLOUD_FILTER = 20

def get_ndvi_composite(start, end):
    """Return median Sentinel-2 NDVI composite for a date range."""
    collection = (
        ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
        .filterBounds(Gotland)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
    )
    print(f"{start} to {end}: {collection.size().getInfo()} images")
    return collection.median().normalizedDifference(['B8', 'B4']).rename('NDVI')

ndvi_before = get_ndvi_composite(START_1, END_1)
ndvi_after = get_ndvi_composite(START_2, END_2)
ndvi_change = ndvi_after.subtract(ndvi_before).rename('NDVI_change')

change_vis = {'min': -0.5, 'max': 0.5, 'palette': ['red', 'white', 'blue']}

Map.addLayer(ndvi_change.clip(Gotland), change_vis, 'NDVI change (2026 - 2020)')
Map.add_colorbar(change_vis, label='NDVI change')
Map.addLayerControl()
Map

 # Watch the world burn
### Inspect the area before fires

In [ ]:
import ee
import geemap

roi = ee.Geometry.Rectangle([-7.8, 42.2, -7.0, 42.6])

pre_start = '2025-06-01'
pre_end   = '2025-06-30'
CLOUD_FILTER = 20

def get_nbr_composite(start, end):
    s2 = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
          .filterBounds(roi)
          .filterDate(start, end)
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER)))
    print(f"{start} to {end}: {s2.size().getInfo()} images")
    return s2.median().normalizedDifference(['B8', 'B12']).rename('NBR')

nbr_pre = get_nbr_composite(pre_start, pre_end)

nbr_vis = {'min': 0, 'max': 1, 'palette': ['brown', 'yellow', 'green']}

# 1. Create the map FIRST
Map = geemap.Map(center=[42.4, -7.4], zoom=9)

# 2. Then swap the basemap
Map.layers = Map.layers[1:]
Map.add_tile_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    name="Esri Dark Gray",
    attribution="Esri",
)

# 3. Then add your data on top
Map.addLayer(nbr_pre.clip(roi), nbr_vis, 'NBR Pre-fire (June 2025)')
Map.addLayerControl()
Map

 ### After fires

In [ ]:
def make_dark_map(center, zoom):
    m = geemap.Map(center=center, zoom=zoom)
    m.layers = m.layers[1:]   # drop default OSM layer
    m.add_tile_layer(
        url="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
        name="Esri Dark Gray",
        attribution="Esri",
    )
    return m

# --- Post-fire period ---
post_start = '2025-08-15'
post_end   = '2025-09-15'

nbr_post = get_nbr_composite(post_start, post_end)

# --- Burn severity: pre minus post ---
dnbr = nbr_pre.subtract(nbr_post).rename('dNBR')

# --- Visualization ---
nbr_vis  = {'min': 0, 'max': 1, 'palette': ['brown', 'yellow', 'green']}
dnbr_vis = {'min': 0, 'max': 1, 'palette': ['white', 'yellow', 'orange', 'red', 'black']}

Map = make_dark_map([42.4, -7.4], 9)
Map.addLayer(nbr_pre.clip(roi),  nbr_vis, 'NBR Pre-fire (Jun 2025)', False)
Map.addLayer(nbr_post.clip(roi), nbr_vis, 'NBR Post-fire (Aug–Sep 2025)', False)

# Hide unburned pixels (dNBR below 0.1) so only burned areas show up
Map.addLayer(dnbr.updateMask(dnbr.gt(0.1)).clip(roi), dnbr_vis, 'dNBR Burn Severity')

Map.add_colorbar(dnbr_vis['palette'], vmin=0, vmax=1, label='Burn Severity (dNBR)')
Map.addLayerControl()
Map

 ## Task 5 Investigate natural hazards
Pick some major event that you are interested in and analyze it using change detection. Some suggestions are:

### Eruption at La Palma 2021-09-19
There was an eruption at La Palma on 2021-09-19. Your task is to use Sentinel-2 imagery to detect vegetation loss and burned/lava-affected areas using NDVI and NBR from Sentinel-2 imagery. The eruption lasted for two months.

a) Inspect NDVI change: Identify areas with strong vegetation loss.  
b) Inspect NBR change: Identify lava and burned areas.

### Nepal Debris Avalanche and Flash Flood 2026-08-26
On Wednesday morning (local time), 26 August 2026, a catastrophic debris flow and flood was likely triggered by a rapid slope failure involving a glacier in Langtang National Park, Nepal, near the border with China. This debris flow and flooding traveled nearly 100 km and impacted populated areas downstream along the Lende Khola and Trishuli Khola rivers, including the Rasuwagadhi border post between China and Nepal.

# Ship detection
Ship detection uses satellite images to find vessels at sea. It supports maritime security, fisheries control, pollution response, and the monitoring of trade routes such as the Strait of Hormuz, through which a large share of the world’s seaborne oil passes. Because many ships switch off their AIS transponders or never carry one, satellites offer an independent view that doesn’t depend on ships reporting their own position.


 Russian warships from the Soviet era have red decks, which makes them easy to pick out using NDVI, even from low-resolution Sentinel-2 imagery.

In [ ]:
import ee
import geemap

# Severomorsk
roi = ee.Geometry.Polygon([
    [33.39, 69.01],
    [33.47, 69.01],
    [33.47, 69.09],
    [33.39, 69.09]
])
ndwi = image.normalizedDifference(['B3','B8']).rename('NDWI')

water_mask = ndwi.gt(-0.2)
# Get a single Sentinel-2 image
image = (ee.ImageCollection('COPERNICUS/S2_SR')
          .filterBounds(roi)
          .filterDate('2024-06-01', '2024-07-31')
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
          .sort('CLOUDY_PIXEL_PERCENTAGE')
          .first())

# Detect red pixels: B4 (red) > 0.1
red_index = image.normalizedDifference(['B4','B3']).rename('Red_Index')
bright_red_mask = image.select('B4').divide(10000).gt(0.1)
ships_candidates = bright_red_mask.And(red_index.gt(0.1)).updateMask(water_mask)

# Visualization
rgb = image.select(['B4','B3','B2']).divide(10000)
Map = geemap.Map(center=[69.085, 33.43], zoom=15)
Map.addLayer(rgb, {'min':0, 'max':0.3}, 'True color')
Map.addLayer(ships_candidates.updateMask(ships_candidates), {'palette':['red']}, 'Red Candidate Pixels')
Map


 ## Ship detection in the Strait of Hormuz
A bit more sophisticated approach is to use Sentinel-1 radar (SAR), which is the standard tool for this. It sees through clouds and at night, and steel ships are very bright against a dark sea. Optical imagery like Sentinel-2 works too, but it is often blocked by haze and only sees ships in daylight.

The approach has four steps:

1. Take one SAR scene for a chosen date.
2. Mask out land.
3. Find very bright pixels, which are ship candidates.
4. Remove anything that is also bright in a long-term median, since that is a fixed object (oil platform, island, buoy, coastline) and not a moving ship.

In [ ]:
import ee
import geemap

# Strait of Hormuz (west, south, east, north)
roi = ee.Geometry.Rectangle([56.0, 26.0, 56.9, 26.8])

THRESHOLD = -18     # dB, tune this (see notes below)
scene_start, scene_end = '2026-02-10', '2026-02-20'   # 10-day window to find a pass
bg_start, bg_end       = '2026-01-01', '2026-02-10'   # several weeks of earlier scenes

s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(roi)
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .select('VH'))

scene = s1.filterDate(scene_start, scene_end)
print("Images in scene window:", scene.size().getInfo())
print("Images in background:", s1.filterDate(bg_start, bg_end).size().getInfo())

scene_img = scene.mosaic().clip(roi)
background = s1.filterDate(bg_start, bg_end).median().clip(roi)

# Sea mask (JRC water occurrence), shrunk a bit to avoid coastline echoes
water = (ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
         .gt(80).unmask(0).focalMin(60, 'circle', 'meters'))

# Candidates: bright now, over water, NOT bright in the long-term median
bright = scene_img.gt(THRESHOLD)
static = background.gt(THRESHOLD).focalMax(30, 'circle', 'meters')
candidates = bright.And(water).And(static.Not()).selfMask().rename('ship')

# Drop single-pixel speckle: keep blobs of at least 3 connected pixels
candidates = candidates.updateMask(candidates.connectedPixelCount(50, True).gte(3))

# Convert blobs to points
ships = candidates.reduceToVectors(
    geometry=roi, scale=10, geometryType='centroid',
    maxPixels=1e10, tileScale=4,
)
print("Ship candidates:", ships.size().getInfo())

# --- Map ---
Map = geemap.Map(center=[26.4, 56.45], zoom=10)
Map.layers = Map.layers[1:]
Map.add_tile_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    name="Esri Dark Gray",
    attribution="Esri",
)
Map.addLayer(scene_img, {'min': -30, 'max': 0}, 'Sentinel-1 VH (dB)')
Map.addLayer(ships.map(lambda f: f.buffer(200)), {'color': 'red'}, 'Ship candidates')
Map.addLayerControl()
Map

 Note the large number of false positives around the coast. This is to be expected. You can read more about ship detection using Sentinel-2 [here](https://www.sciencedirect.com/science/article/pii/S2666675826001141). The main downside is that small ships are not visible, and there is a long revisit time. Sentinel-1 passes only every few days, so the result is a snapshot, not continuous tracking.

 ### Task 6 Compare ships in the Strait of Hormuz
Compare pre-war levels from February 22–28, 2026 with March 1–7, 2026.

 ### Snow monitor
This section is only for inspiration and there are no more assignments to complete.

While developing this module, I also built a side project for my kids where they can follow spring from space using satellite imagery. It's a small script that runs on a Raspberry Pi and simply pulls the latest satellite imagery of our village every day and displays it on an old monitor that I set up in our kitchen. You can use this for anything from monitoring ships in a harbor to active volcanoes. This is what you need:

* Some easily impressed kids.
* An old monitor.
* An old computer or something equivalent to a Raspberry Pi.
* An idea of what to spy on.

Modify the code below so the coordinates cover the area you want to spy on.

Copy the code and save it to a script that you call something like `spy.py`. Run the script. The while loop will look for the latest satellite imagery once every day and then display that image on the monitor.

In [ ]:
import ee
import geemap
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import time

# Authenticate and initialize Google Earth Engine API
ee.Authenticate()

# Change 'my-project' to your actual project
ee.Initialize(project='ee-williamlidberg')

# Set the location (bounding box)
roi = ee.Geometry.Rectangle([19.75, 63.9, 20, 64.03])

# Define the number of days to look back
num_days_to_look_back = 5
plt.figure(figsize=(20, 20))

while True:  # Run indefinitely
    latest_image = None
    latest_acquisition_time = None
    for i in range(num_days_to_look_back):
        start_date = datetime.today() - timedelta(days=i + 1)
        end_date = datetime.today() - timedelta(days=i)
        collection = (ee.ImageCollection('COPERNICUS/S2')
                      .filterBounds(roi)
                      .filterDate(start_date, end_date)
                      .sort('system:time_start', False)  # Sort in descending order to get the latest image first
                      )
        # Get the latest image for the current day
        image = collection.first()
        if image.getInfo():
            image_array = geemap.ee_to_numpy(image.select(['B4', 'B3', 'B2']).divide(4000), region=roi)
            if image_array.size > 0:
                latest_image = image
                latest_acquisition_time = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd HH:mm:ss').getInfo()
    # Check if the latest image and acquisition time are available
    if latest_image is not None and latest_acquisition_time is not None:
        center = roi.centroid().coordinates().reverse().getInfo()
        zoom = 12  # Adjust the zoom level as needed

        plt.clf()
        image_array = geemap.ee_to_numpy(latest_image.select(['B4', 'B3', 'B2']).divide(4000), region=roi)
        if image_array.size > 0:
            plt.imshow(image_array, vmin=0, vmax=0.4, aspect='equal')
            plt.title(f'True Color Image - {latest_acquisition_time}')
            plt.axis('off')

        print(f'Acquisition Time of the Latest Image: {latest_acquisition_time}')
        plt.pause(1)
        print('Updated image')

    time.sleep(86400) # number of seconds in a day


## References
  * Wu, Q., (2020). geemap: A Python package for interactive mapping with Google Earth Engine. The Journal of Open Source Software, 5(51), 2305. https://doi.org/10.21105/joss.02305
  * Wu, Q., Lane, C. R., Li, X., Zhao, K., Zhou, Y., Clinton, N., DeVries, B., Golden, H. E., & Lang, M. W. (2019). Integrating LiDAR data and multi-temporal aerial imagery to map wetland inundation dynamics using Google Earth Engine. Remote Sensing of Environment, 228, 1-13. https://doi.org/10.1016/j.rse.2019.04.015 (pdf | source code)
  * Cao, Y., Wang, H., Su, F., Fu, D., Yan, F., Hong, X., He, W., Xu, N., Ou, Y., & Chen, C. (2026). Satellite radar and AIS reveal a 97% decline in shipping traffic through the Strait of Hormuz. The Innovation, 7(6), Article 101367.